# ML-09 — Validation and Research Claim Audit

**Davidson C. Briones · FlyRank ML Internship · Week 6**

[Open in Colab](https://colab.research.google.com/github/Davidsoooooon/ML-Intern---Briones/blob/main/work/notebooks/w06_validation_audit.ipynb)

**Completed run — 21 November 2026:** all nine notebook self-checks passed, with no recorded execution errors. The saved Colab outputs include the model comparisons, nested client-grouped validation, leakage controls, failure examples, and revised claim.

The audit uses 75,526 eligible pages from 37 pseudonymized clients. Features cover March 1–14, 2026; the decline proxy covers March 15–28. Saved Week-5 observations are labeled historical and are distinguished from the new audit results.

These are directional development observations for editorial decision support. They do not establish a causal refresh effect or performance in a new month.

**Reproducing the run:** open this notebook in Colab, enable access to your existing `HF_TOKEN` secret, and select **Runtime → Run all**. The first run downloads the approved March partition; later runs reuse the local cache.

**Provenance:** the paper and original Week-5 notebook were inspected at commit `3b5601ce66bc329707683284dc65ed9ed8da1d13`. The audit follows the repository's validation and data-use guidance. Prepared with AI assistance; the reported model results come from the saved Colab execution.


## 1. Two paper findings + my methodology questions

Source: FlyRank, *The State of AI-Driven SEO in Numbers*, March 2026,
[`docs/flyrank-seo-research-march-2026.pdf`](../../docs/flyrank-seo-research-march-2026.pdf).
The paper leads with portfolio comparisons and explicitly treats its ML appendix as exploratory.
The questions below extend those disclosed limits; they are not a grade of the paper.

### Finding 1 — Health-score feature importance (printed page 27)

The Random Forest assigns 43% importance to average position, 32% to impressions, and 15%
to scroll depth. Pages 5 and 36 define health as a composite of impressions, position, CTR,
and scroll depth. Page 27 already notes this overlap and limits the interpretation to model behavior.

**My methodology question:** Could the label recipe and feature windows be shown beside the
importance chart? If the purpose is to approximate the health score, component inputs are
appropriate, but their importance cannot independently validate external search performance.
For a predictive claim, how would the result change when the target is a later observed outcome
and score components from that later window are excluded? I would also ask for a remove-the-suspect
experiment before interpreting any importance as a useful signal.

**What I apply to my work:** distinguish a measured future-impression proxy from an editorial
decision, exclude future measurements, and deliberately inject a forbidden target-derived feature
to check that my evaluation detects the leakage.

### Finding 2 — Growth classification (printed page 29)

The exploratory Logistic Regression reports 71% holdout accuracy for growing versus declining
pages. Page 36 specifies an 80/20 split on an active-content subset; it does not specify a
client-disjoint or forward-time holdout. Page 5 defines growth/decline using a 30-day comparison.

**My methodology question:** How were stable, flat, and new pages handled when constructing
the binary label, and what was the majority-class rate in that exact holdout? Were feature
windows strictly earlier than the label window, and were all pages from each client kept on
one side of the split? A grouped comparison with its base rate and a later-window evaluation
would clarify which generalization claims the 71% supports. An unspecified split is a question
to resolve, not evidence that leakage occurred.

**What I apply to my work:** show base rates, separate model selection from outer evaluation,
compare random-row and client-held-out estimates, and retain a March-only limitation. The paper's
10% trend definition, the starter CSV's 20% rule, and my 14-day 20% proxy are different tasks;
their headline metrics are not directly comparable.

In [10]:
# Install only missing dependencies. Colab already supplies most of these.
import importlib.util
import subprocess
import sys

packages = {
    "numpy": "numpy>=1.26", "pandas": "pandas>=2.2",
    "sklearn": "scikit-learn>=1.4", "duckdb": "duckdb>=1.0",
    "huggingface_hub": "huggingface_hub>=0.24",
    "pyarrow": "pyarrow", "IPython": "ipython",
}
missing = [spec for module, spec in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import os
import json
import hashlib
import platform
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import duckdb
import sklearn
from IPython.display import display, Markdown
from huggingface_hub import hf_hub_download, get_token
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
FEATURES = ["impressions_early", "avg_position_early",
            "position_volatility_early", "active_days_early"]
TARGET = "is_declining_proxy"
GROUP = "client_hash_id"
KEYS = [GROUP, "content_hash_id"]
DATASET = "FlyRank/internship-warehouse"
DATA_REVISION = "50cbf7c3909d07be4d1b5906b4d09e882e5acbf2"
PARTITION = "fact_content_daily_performance/month=2026-03/data_0.parquet"
FEATURE_END = pd.Timestamp("2026-03-14")
DECISION_DATE = pd.Timestamp("2026-03-15")
LABEL_START = pd.Timestamp("2026-03-15")
LABEL_END = pd.Timestamp("2026-03-28")
assert FEATURE_END < DECISION_DATE <= LABEL_START <= LABEL_END

checks = {"data": False, "grain": False, "comparison": False,
          "nested_cv": False, "leakage": False, "errors": False,
          "claim": False}
model_data = None
print("Environment:", {"python": platform.python_version(), "pandas": pd.__version__,
      "numpy": np.__version__, "sklearn": sklearn.__version__, "duckdb": duckdb.__version__})
print("Execution started (UTC):", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Feature dates: March 1–14; proxy dates: March 15–28, 2026.")

Environment: {'python': '3.13.18', 'pandas': '2.2.3', 'numpy': '2.4.6', 'sklearn': '1.6.1', 'duckdb': '1.3.2'}
Execution started (UTC): 2026-11-21T04:08:21+00:00
Feature dates: March 1–14; proxy dates: March 15–28, 2026.


## 2. My model under an honest split (before/after)

### What Week 5 actually did

[`w05_model.ipynb`](w05_model.ipynb) already used `GroupShuffleSplit(test_size=0.20,
random_state=42)`: 29 training clients and 8 test clients, with no client overlap.
It used four early-window features and a training-fitted Week-4 rule.
I do **not** describe Week 5 as a random split.

The substantive weakness is that `best_model_name` was chosen using **test Precision@50**.
The same test set was then used to describe the winner. This is exploratory model selection,
not an untouched final estimate. A fixed seed also does not measure variation across clients.

The table below is transcribed from the saved Week-5 outputs, rounded to three decimals.
It is shown even if warehouse access is unavailable and is explicitly labeled historical.

In [11]:
historical = pd.DataFrame([
    ["Week-4 baseline rule", 0.350, 0.240, 0.265, 0.500, 0.265],
    ["Logistic Regression", 0.500, 0.480, 0.258, 0.480, 0.265],
    ["Random Forest", 0.200, 0.280, 0.289, 0.536, 0.265],
], columns=["Method", "Precision@20", "Precision@50", "Average precision", "ROC-AUC", "Test base rate"])
print("HISTORICAL — saved Week-5 output, not a new run")
print("75,526 pages; 37 clients; train 65,274 pages; test 10,252 pages.")
display(historical.round(3))
print("At 50 reviews, the recorded LR queue contained 24 proxy positives and 26 negatives.")

HISTORICAL — saved Week-5 output, not a new run
75,526 pages; 37 clients; train 65,274 pages; test 10,252 pages.


,Method,Precision@20,Precision@50,Average precision,ROC-AUC,Test base rate
0,Week-4 baseline rule,0.35,0.24,0.265,0.500,0.265
1,Logistic Regression,0.50,0.48,0.258,0.480,0.265
2,Random Forest,0.20,0.28,0.289,0.536,0.265


At 50 reviews, the recorded LR queue contained 24 proxy positives and 26 negatives.


### Prespecified audit design

1. **Re-run the original grouped holdout.** Keep the four features, proxy, seed, model
   settings, and training-fitted rule. Sort by pseudonymous keys for reproducible row order.
   Week 5 did not order its SQL result, and library versions may differ; exact bitwise
   reproduction is not assumed.
2. **Random-row diagnostic → grouped holdout.** Fit the same fixed Logistic Regression on
   both splits. Match test row counts, display group overlap and prevalence, and avoid choosing
   a favorable seed. Different test populations mean the gap is not a causal estimate of leakage
   or a measurement of memorization by itself.
3. **Improvement over Week 5: nested grouped validation.** Use five outer client-disjoint
   folds. Within each outer training set, choose between the two unchanged Week-5 models with
   three inner grouped folds using mean Precision@50; exact ties favor Logistic Regression.
   Only then evaluate on the outer fold. Refit preprocessing and baseline thresholds on training
   data each time. Display every fold, the mean, and the range; folds are not independent trials
   and the range is not a confidence interval.

Precision@50 is the decision metric. Average precision and ROC-AUC describe the wider ranking.
The held-out positive rate is the expected precision of a random ranking; ROC-AUC's random
reference is 0.5. The rule baseline is evaluated on exactly the same held-out rows as the models.
A queue here covers 50 pages across the held-out clients, not 50 pages per client.

Nested validation reduces direct test-set model selection in this rerun, but March data have
already informed earlier work. It remains a retrospective development audit. A genuinely new
calendar period is still needed for confirmation; separated feature/label windows are not themselves
a forward-time train/test experiment.

In [12]:
def resolve_token():
    # Never display the value or include it in notebook source/output.
    token = get_token()
    if token:
        return token
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

local_partition = os.getenv("FLYRANK_MARCH_PARQUET")
token = resolve_token()
data_file = None
if local_partition:
    data_file = Path(local_partition).expanduser().resolve()
    if not data_file.is_file():
        raise FileNotFoundError("FLYRANK_MARCH_PARQUET must point to the approved March partition.")
elif token:
    try:
        data_file = Path(hf_hub_download(
            repo_id=DATASET, repo_type="dataset", filename=PARTITION,
            revision=DATA_REVISION, token=token,
        ))
    except Exception:
        raise RuntimeError(
            "Warehouse download failed. Confirm gate approval and notebook access to your "
            "read-only HF_TOKEN secret, then run again. No credential details are printed."
        ) from None
else:
    # Reuse only an existing official Hub cache, without a network request.
    try:
        data_file = Path(hf_hub_download(
            repo_id=DATASET, repo_type="dataset", filename=PARTITION,
            revision=DATA_REVISION, local_files_only=True,
        ))
    except Exception:
        print("NOT RUN — March warehouse data unavailable. In Colab, enable access to your existing HF_TOKEN secret and Run all.")
token = None

if data_file is not None:
    con = duckdb.connect()
    con.read_parquet(str(data_file)).create_view("march_daily")
    raw_summary = con.sql("""
        SELECT COUNT(*) AS rows, MIN(report_date) AS first_date,
               MAX(report_date) AS last_date,
               COUNT(DISTINCT client_hash_id) AS clients,
               COUNT(*) FILTER (WHERE client_hash_id IS NULL OR content_hash_id IS NULL
                                OR report_date IS NULL) AS null_keys,
               COUNT(*) FILTER (WHERE gsc_impressions IS NULL OR gsc_impressions < 0) AS invalid_impressions
        FROM march_daily
    """).df()
    display(raw_summary)
    assert pd.Timestamp(raw_summary.loc[0, "first_date"]) == pd.Timestamp("2026-03-01")
    assert pd.Timestamp(raw_summary.loc[0, "last_date"]) == pd.Timestamp("2026-03-31")
    assert raw_summary.loc[0, "null_keys"] == 0
    assert raw_summary.loc[0, "invalid_impressions"] == 0
    duplicate_keys = con.sql("""
        SELECT COUNT(*) FROM (
            SELECT report_date, client_hash_id, content_hash_id
            FROM march_daily GROUP BY ALL HAVING COUNT(*) > 1
        )
    """).fetchone()[0]
    assert duplicate_keys == 0, "Duplicate page-day rows: stop before aggregation."
    checks["grain"] = True

    model_data = con.sql("""
        WITH page_windows AS (
            SELECT client_hash_id, content_hash_id,
              SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
                       THEN gsc_impressions ELSE 0 END) AS impressions_early,
              AVG(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
                       THEN NULLIF(gsc_avg_position, 0) END) AS avg_position_early,
              STDDEV_SAMP(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
                              THEN NULLIF(gsc_avg_position, 0) END) AS position_volatility_early,
              COUNT(DISTINCT CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
                                  AND gsc_impressions > 0 THEN report_date END) AS active_days_early,
              SUM(CASE WHEN report_date BETWEEN DATE '2026-03-15' AND DATE '2026-03-28'
                       THEN gsc_impressions ELSE 0 END) AS impressions_late,
              COUNT(DISTINCT CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
                                  THEN report_date END) AS observed_days_early,
              COUNT(DISTINCT CASE WHEN report_date BETWEEN DATE '2026-03-15' AND DATE '2026-03-28'
                                  THEN report_date END) AS observed_days_late
            FROM march_daily
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-28'
            GROUP BY client_hash_id, content_hash_id
        )
        SELECT *, CAST(impressions_late < 0.80 * impressions_early AS INTEGER) AS is_declining_proxy
        FROM page_windows WHERE impressions_early >= 100
    """).df().sort_values(KEYS, kind="stable").reset_index(drop=True)
    coverage = con.sql("""
        SELECT client_hash_id,
          COUNT(DISTINCT CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
                              THEN report_date END) AS early_days,
          COUNT(DISTINCT CASE WHEN report_date BETWEEN DATE '2026-03-15' AND DATE '2026-03-28'
                              THEN report_date END) AS late_days
        FROM march_daily GROUP BY client_hash_id
    """).df()
    relevant_coverage = coverage[coverage[GROUP].isin(model_data[GROUP])]
    print("Client fact-day coverage (availability diagnostic, not proof of complete tracking):")
    display(relevant_coverage.groupby(["early_days", "late_days"]).size().rename("clients").reset_index())
    print("Pages with no observed late-window daily row:", int((model_data.observed_days_late == 0).sum()))
    print("Feature missingness:")
    display(model_data[FEATURES].isna().mean().rename("missing_fraction").to_frame())
    assert not model_data.duplicated(KEYS).any()
    assert not model_data[KEYS].isna().any().any()
    assert model_data[TARGET].nunique() == 2
    assert model_data[GROUP].nunique() >= 10
    assert np.isfinite(model_data[FEATURES].dropna().to_numpy()).all()
    assert (model_data.impressions_early >= 100).all()
    assert model_data.active_days_early.between(1, 14).all()
    assert np.array_equal(model_data[TARGET],
                          (model_data.impressions_late < 0.8 * model_data.impressions_early).astype(int))
    digest = hashlib.sha256()
    with data_file.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    print("Source revision:", DATA_REVISION)
    print("Partition SHA256:", digest.hexdigest())
    print("Pages:", len(model_data), "Clients:", model_data[GROUP].nunique(),
          "Proxy-positive rate:", round(model_data[TARGET].mean(), 4))
    if (len(model_data), model_data[GROUP].nunique()) != (75526, 37):
        raise ValueError("Cohort differs from Week 5 (75,526 pages / 37 clients). Investigate before claiming a like-for-like rerun.")
    checks["data"] = True

,rows,first_date,last_date,clients,null_keys,invalid_impressions
0,9841378,2026-03-01,2026-03-31,55,0,0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Client fact-day coverage (availability diagnostic, not proof of complete tracking):


,early_days,late_days,clients
0,12,14,1
1,14,14,36


Pages with no observed late-window daily row: 0
Feature missingness:


,missing_fraction
impressions_early,0.000000
avg_position_early,0.000000
position_volatility_early,0.000093
active_days_early,0.000000


Source revision: 50cbf7c3909d07be4d1b5906b4d09e882e5acbf2
Partition SHA256: 6ebd367f05932238df05b193f9cfb9cc7839d7856b65ef74b5a122d5fadef589
Pages: 75526 Clients: 37 Proxy-positive rate: 0.3379


In [13]:
def make_models():
    return {
        "Logistic Regression": make_pipeline(
            SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
            LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)),
        "Random Forest": make_pipeline(
            SimpleImputer(strategy="median", add_indicator=True),
            RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1)),
    }

def ranks(scores):
    scores = np.asarray(scores, dtype=float)
    assert np.isfinite(scores).all()
    # Stable input order is client/content key order; no outcome is used to break ties.
    return np.argsort(-scores, kind="stable")

def precision_at_k(y, scores, k=50):
    assert len(y) == len(scores) and len(y) >= k
    return float(np.asarray(y)[ranks(scores)[:k]].mean())

def metrics(y, scores):
    y = np.asarray(y, dtype=int)
    assert np.unique(y).size == 2
    return {"Precision@20": precision_at_k(y, scores, 20),
            "Precision@50": precision_at_k(y, scores, 50),
            "Average precision": average_precision_score(y, scores),
            "ROC-AUC": roc_auc_score(y, scores),
            "Test base rate": float(y.mean()), "Test pages": len(y)}

def baseline_scores(train, test):
    def percentile(column):
        v = pd.to_numeric(train[column], errors="coerce").dropna().sort_values().to_numpy()
        if not len(v):
            raise ValueError("No training observations for baseline column.")
        q = pd.to_numeric(test[column], errors="coerce").fillna(-np.inf).to_numpy()
        return np.searchsorted(v, q, side="right") / len(v)
    hit = ((test.impressions_early >= train.impressions_early.quantile(.75)) &
           (test.position_volatility_early >= train.position_volatility_early.quantile(.75)))
    return np.where(hit, percentile("impressions_early") * percentile("position_volatility_early"), 0.0)

def group_assertions(train, test):
    assert set(train[GROUP]).isdisjoint(test[GROUP])
    assert set(map(tuple, train[KEYS].to_numpy())).isdisjoint(map(tuple, test[KEYS].to_numpy()))
    assert train[TARGET].nunique() == test[TARGET].nunique() == 2
    assert len(test) >= 50

if checks["data"]:
    splitter = GroupShuffleSplit(n_splits=1, test_size=.20, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(model_data[FEATURES], model_data[TARGET], groups=model_data[GROUP]))
    train_data = model_data.iloc[train_idx].copy()
    test_data = model_data.iloc[test_idx].copy()
    group_assertions(train_data, test_data)
    holdout_models, holdout_scores = {}, {"Week-4 baseline rule": baseline_scores(train_data, test_data)}
    for name, model in make_models().items():
        model.fit(train_data[FEATURES], train_data[TARGET])
        holdout_models[name] = model
        holdout_scores[name] = model.predict_proba(test_data[FEATURES])[:, 1]
    rerun = pd.DataFrame([{"Method": name, **metrics(test_data[TARGET], score)}
                          for name, score in holdout_scores.items()])
    print("NEW RUN — original grouped design; model selection still treated as development")
    display(rerun.round(4))
    display(historical.merge(rerun, on="Method", suffixes=(" (saved W5)", " (rerun)"))[
        ["Method", "Precision@50 (saved W5)", "Precision@50 (rerun)",
         "Test base rate (saved W5)", "Test base rate (rerun)"]].round(4))

    random_train, random_test = train_test_split(np.arange(len(model_data)),
        test_size=len(test_data), stratify=model_data[TARGET], random_state=RANDOM_STATE)
    random_train, random_test = np.sort(random_train), np.sort(random_test)
    random_model = make_models()["Logistic Regression"]
    random_model.fit(model_data.iloc[random_train][FEATURES], model_data.iloc[random_train][TARGET])
    random_scores = random_model.predict_proba(model_data.iloc[random_test][FEATURES])[:, 1]
    diagnostic = pd.DataFrame([
        {"Split": "Random rows (diagnostic only)",
         "Shared clients": len(set(model_data.iloc[random_train][GROUP]) & set(model_data.iloc[random_test][GROUP])),
         **metrics(model_data.iloc[random_test][TARGET], random_scores)},
        {"Split": "Client holdout", "Shared clients": 0,
         **metrics(test_data[TARGET], holdout_scores["Logistic Regression"])},
    ])
    print("SPLIT COMPARISON — same fixed LR settings; matched test-row count")
    display(diagnostic.round(4))
    gap = diagnostic.loc[1, "Precision@50"] - diagnostic.loc[0, "Precision@50"]
    print(f"Observed grouped minus random P@50: {gap:+.3f}; test populations differ.")
    print("This gap alone does not identify its cause or establish memorization.")
    checks["comparison"] = True
else:
    print("NOT RUN — original-design rerun and split comparison require warehouse data.")

NEW RUN — original grouped design; model selection still treated as development


,Method,Precision@20,Precision@50,Average precision,ROC-AUC,Test base rate,Test pages
0,Week-4 baseline rule,0.35,0.24,0.2651,0.4996,0.2645,10252
1,Logistic Regression,0.50,0.48,0.2583,0.4799,0.2645,10252
2,Random Forest,0.20,0.30,0.2871,0.5335,0.2645,10252


,Method,Precision@50 (saved W5),Precision@50 (rerun),Test base rate (saved W5),Test base rate (rerun)
0,Week-4 baseline rule,0.24,0.24,0.265,0.2645
1,Logistic Regression,0.48,0.48,0.265,0.2645
2,Random Forest,0.28,0.30,0.265,0.2645


SPLIT COMPARISON — same fixed LR settings; matched test-row count


,Split,Shared clients,Precision@20,Precision@50,Average precision,ROC-AUC,Test base rate,Test pages
0,Random rows (diagnostic only),31,0.45,0.40,0.4150,0.5883,0.3379,10252
1,Client holdout,0,0.50,0.48,0.2583,0.4799,0.2645,10252


Observed grouped minus random P@50: +0.080; test populations differ.
This gap alone does not identify its cause or establish memorization.


In [14]:
if checks["comparison"]:
    outer = GroupKFold(n_splits=5)
    cv_rows, selection_rows, oof_parts, outer_models = [], [], [], []
    seen = np.zeros(len(model_data), dtype=int)
    for fold, (outer_train_idx, outer_test_idx) in enumerate(
        outer.split(model_data[FEATURES], model_data[TARGET], model_data[GROUP]), start=1
    ):
        train = model_data.iloc[outer_train_idx].copy()
        test = model_data.iloc[outer_test_idx].copy()
        group_assertions(train, test)
        inner_scores = {name: [] for name in make_models()}
        inner = GroupKFold(n_splits=3)
        for inner_train_idx, inner_valid_idx in inner.split(train[FEATURES], train[TARGET], train[GROUP]):
            inner_train, inner_valid = train.iloc[inner_train_idx], train.iloc[inner_valid_idx]
            group_assertions(inner_train, inner_valid)
            for name, estimator in make_models().items():
                estimator.fit(inner_train[FEATURES], inner_train[TARGET])
                score = estimator.predict_proba(inner_valid[FEATURES])[:, 1]
                inner_scores[name].append(precision_at_k(inner_valid[TARGET], score))
        means = {name: float(np.mean(values)) for name, values in inner_scores.items()}
        # dict order breaks exact ties in favor of Logistic Regression, as prespecified.
        selected = max(means, key=means.get)
        selection_rows.append({"Outer fold": fold, "Selected model": selected,
                               **{name + " inner P@50": value for name, value in means.items()}})
        scores = {"Week-4 baseline rule": baseline_scores(train, test)}
        fitted = {}
        for name, estimator in make_models().items():
            estimator.fit(train[FEATURES], train[TARGET])
            fitted[name] = estimator
            scores[name] = estimator.predict_proba(test[FEATURES])[:, 1]
        scores["Inner-selected model"] = scores[selected]
        for name, score in scores.items():
            cv_rows.append({"Fold": fold, "Method": name,
                            "Train clients": train[GROUP].nunique(),
                            "Test clients": test[GROUP].nunique(),
                            **metrics(test[TARGET], score)})
        # Keep small model objects in memory, never export the underlying dataset.
        outer_models.append((fold, fitted[selected], test.index.to_numpy()))
        part = test.copy()
        part["fold"] = fold
        part["selected_model"] = selected
        part["model_score"] = scores[selected]
        part["baseline_score"] = scores["Week-4 baseline rule"]
        order = ranks(part.model_score)
        within_fold_rank = np.empty(len(part), dtype=int)
        within_fold_rank[order] = np.arange(1, len(part) + 1)
        part["queue_rank"] = within_fold_rank
        oof_parts.append(part)
        seen[outer_test_idx] += 1
        print(f"Completed outer fold {fold}/5; selection used inner folds only.")
    assert (seen == 1).all()
    cv_results = pd.DataFrame(cv_rows)
    oof = pd.concat(oof_parts).sort_index()
    cv_summary = cv_results.groupby("Method").agg(
        mean_p50=("Precision@50", "mean"), min_p50=("Precision@50", "min"),
        max_p50=("Precision@50", "max"), mean_ap=("Average precision", "mean"),
        mean_auc=("ROC-AUC", "mean"), mean_test_base_rate=("Test base rate", "mean"),
    ).reset_index()
    display(pd.DataFrame(selection_rows).round(4))
    display(cv_results.round(4))
    display(cv_summary.round(4))
    paired = cv_results.pivot(index="Fold", columns="Method", values="Precision@50")
    paired["Selected minus baseline"] = paired["Inner-selected model"] - paired["Week-4 baseline rule"]
    display(paired.round(4))
    mean_delta = paired["Selected minus baseline"].mean()
    print(f"Measured mean within-fold P@50 difference vs rule: {mean_delta:+.3f}.")
    print("Means are fold means (50 reviews per outer fold), not a pooled top-50 ranking.")
    print("Fold scores are descriptive variability; they are not a confidence interval or a new-month test.")
    checks["nested_cv"] = True
else:
    print("NOT RUN — nested grouped validation requires warehouse data.")

Completed outer fold 1/5; selection used inner folds only.
Completed outer fold 2/5; selection used inner folds only.
Completed outer fold 3/5; selection used inner folds only.
Completed outer fold 4/5; selection used inner folds only.
Completed outer fold 5/5; selection used inner folds only.


,Outer fold,Selected model,Logistic Regression inner P@50,Random Forest inner P@50
0,1,Random Forest,0.3133,0.5533
1,2,Random Forest,0.3467,0.4933
2,3,Random Forest,0.4533,0.6600
3,4,Random Forest,0.3267,0.4467
4,5,Random Forest,0.2800,0.5200


,Fold,Method,Train clients,Test clients,Precision@20,Precision@50,Average precision,ROC-AUC,Test base rate,Test pages
0,1,Week-4 baseline rule,36,1,0.70,0.78,0.3576,0.5028,0.3546,17280
1,1,Logistic Regression,36,1,0.20,0.16,0.3285,0.4825,0.3546,17280
2,1,Random Forest,36,1,0.55,0.52,0.4160,0.5978,0.3546,17280
3,1,Inner-selected model,36,1,0.55,0.52,0.4160,0.5978,0.3546,17280
4,2,Week-4 baseline rule,29,8,0.70,0.64,0.4418,0.5013,0.4407,14542
5,2,Logistic Regression,29,8,0.10,0.10,0.4402,0.5148,0.4407,14542
6,2,Random Forest,29,8,1.00,0.74,0.5155,0.5893,0.4407,14542
7,2,Inner-selected model,29,8,1.00,0.74,0.5155,0.5893,0.4407,14542
8,3,Week-4 baseline rule,31,6,0.30,0.42,0.4452,0.5000,0.4446,14606
9,3,Logistic Regression,31,6,0.30,0.46,0.3892,0.4148,0.4446,14606


,Method,mean_p50,min_p50,max_p50,mean_ap,mean_auc,mean_test_base_rate
0,Inner-selected model,0.544,0.40,0.74,0.4060,0.5808,0.3372
1,Logistic Regression,0.300,0.10,0.46,0.3563,0.5203,0.3372
2,Random Forest,0.544,0.40,0.74,0.4060,0.5808,0.3372
3,Week-4 baseline rule,0.504,0.24,0.78,0.3401,0.5033,0.3372


Method,Inner-selected model,Logistic Regression,Random Forest,Week-4 baseline rule,Selected minus baseline
Fold,,,,,
1,0.52,0.16,0.52,0.78,-0.26
2,0.74,0.10,0.74,0.64,0.10
3,0.46,0.46,0.46,0.42,0.04
4,0.60,0.42,0.60,0.44,0.16
5,0.40,0.36,0.40,0.24,0.16


Measured mean within-fold P@50 difference vs rule: +0.040.
Means are fold means (50 reviews per outer fold), not a pooled top-50 ranking.
Fold scores are descriptive variability; they are not a confidence interval or a new-month test.


## 3. Leakage audit

### Feature and label lineage

Prediction date: **March 15, 2026**. Features use March 1–14 only; outcomes use March 15–28.
This retrospective release does not establish when historical GSC values became available or
were revised. An operational version must wait for complete early-window reporting or introduce
a documented reporting lag; timestamp separation alone does not prove real-time availability.

| Field or operation | Role | Decision and remaining risk |
|---|---|---|
| `impressions_early` | Feature; also proxy denominator and eligibility criterion | Knowable before outcome, not derived from the future label. Its role in the relative-change definition can create mathematical coupling/regression to the mean; run an ablation. |
| `avg_position_early` | Feature | Early dates only; zero → missing. Unweighted daily average reproduces Week 5 and is not an impression-weighted position measure. |
| `position_volatility_early` | Feature | Early sample SD; impute missing values using training rows only and add a missingness indicator. |
| `active_days_early` | Feature | Early days with positive impressions only; activity is not proof of complete tracking. |
| `impressions_late`, early/late ratio, percentage change | Label construction/diagnostic only | Future values; excluded from honest models. The deliberately leaky control is isolated and never used in a recommendation. |
| `is_declining_proxy` | Target | 1 if late impressions < 0.80 × early impressions. A decline proxy, not a verified content defect or refresh-response label. |
| Client/content IDs | Context | Split, join, stable tie order only; never model inputs. Group clients, not only individual pages. |
| `observed_days_late`, client late coverage | Measurement audit | Outcome-window context only; never features. Missing late rows must not automatically be interpreted as zero demand. |
| `trend_pct`, `trend_direction`, 90-day aggregates | Excluded | Different starter task and/or overlapping windows; not selected here. |
| Product scores/flags such as `health_score`, `priority_score` | Excluded | Existing decisions are not observations for this model. |
| Median/scaler/quantiles/percentile strengths | Learned preprocessing | Fit afresh on each training partition, including each inner fold. |
| Winner chosen on final test scores | Evaluation leakage risk | Week-5 development choice; replaced with selection inside the outer training data. |

**Coverage limitation:** a page with absent late rows receives a zero sum under the original
Week-5 recipe. The coverage outputs surface this risk without changing the cohort based on
the observed outcome. Client fact-day coverage does not replace checking `dim_clients.gsc_data_start`
and source ingestion completeness. No causal refresh or comprehensive production-validity claim follows.

In [15]:
FORBIDDEN = {TARGET, "impressions_late", "leak_decline_ratio", "observed_days_late",
             GROUP, "content_hash_id", "trend_pct", "trend_direction", "health_score", "priority_score"}
assert set(FEATURES).isdisjoint(FORBIDDEN)
assert len(FEATURES) == len(set(FEATURES)) == 4
print("Static feature allowlist and window assertions: PASS")

if checks["nested_cv"]:
    # Mechanism check on the fixed grouped development holdout, not a publishable model.
    controls = []
    clean_tree = make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                               DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE))
    clean_tree.fit(train_data[FEATURES], train_data[TARGET])
    controls.append({"Experiment": "Clean shallow tree", **metrics(test_data[TARGET],
                    clean_tree.predict_proba(test_data[FEATURES])[:, 1])})
    leaky_train, leaky_test = train_data[FEATURES].copy(), test_data[FEATURES].copy()
    leaky_train["leak_decline_ratio"] = train_data.impressions_late / train_data.impressions_early
    leaky_test["leak_decline_ratio"] = test_data.impressions_late / test_data.impressions_early
    leaky_tree = clone(clean_tree).fit(leaky_train, train_data[TARGET])
    leaky_scores = leaky_tree.predict_proba(leaky_test)[:, 1]
    controls.append({"Experiment": "DELIBERATELY LEAKY — discard", **metrics(test_data[TARGET], leaky_scores)})
    leak_auc = roc_auc_score(test_data[TARGET], leaky_scores)
    display(pd.DataFrame(controls).round(4))
    assert leak_auc >= .99, "Expected near-perfect label reconstruction; investigate the control."
    print("The ratio directly constructs the proxy: a high control score is leakage, not useful performance.")
    del leaky_train, leaky_test, leaky_tree, leaky_scores

    # Ablation keeps the known-before-decision early volume in eligibility/label construction,
    # removing it ONLY as a feature. This probes reliance, not removal of all mathematical coupling.
    reduced = [feature for feature in FEATURES if feature != "impressions_early"]
    ablated = make_models()["Logistic Regression"]
    ablated.fit(train_data[reduced], train_data[TARGET])
    ablated_scores = ablated.predict_proba(test_data[reduced])[:, 1]
    ablation = pd.DataFrame([
        {"Experiment": "Fixed LR, all four early features", **metrics(test_data[TARGET], holdout_scores["Logistic Regression"])},
        {"Experiment": "Fixed LR, early impressions removed", **metrics(test_data[TARGET], ablated_scores)},
    ])
    display(ablation.round(4))
    print("A change in this ablation measures feature dependence; it does not alone prove or disprove leakage.")

    importance_rows = []
    for fold, estimator, test_indices in outer_models:
        heldout = model_data.loc[test_indices]
        importance = permutation_importance(estimator, heldout[FEATURES], heldout[TARGET],
            scoring="average_precision", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
        for feature, mean, sd in zip(FEATURES, importance.importances_mean, importance.importances_std):
            importance_rows.append({"Fold": fold, "Feature": feature, "AP drop mean": mean,
                "Permutation SD": sd, "Test base rate": heldout[TARGET].mean()})
    importance_table = pd.DataFrame(importance_rows)
    display(importance_table.round(4))
    display(importance_table.groupby("Feature").agg(
        mean_ap_drop=("AP drop mean", "mean"), min_fold_drop=("AP drop mean", "min"),
        max_fold_drop=("AP drop mean", "max"), mean_test_base_rate=("Test base rate", "mean")
    ).sort_values("mean_ap_drop", ascending=False).round(4))
    print("Negative AP drops mean shuffling helped in that fold; importances are diagnostic, not causal.")
    print("No features are changed in response to these outer-test diagnostics in this audit.")
    checks["leakage"] = True
else:
    print("NOT RUN — empirical leakage controls, ablation, and held-out importance require data.")

Static feature allowlist and window assertions: PASS


,Experiment,Precision@20,Precision@50,Average precision,ROC-AUC,Test base rate,Test pages
0,Clean shallow tree,0.65,0.56,0.2746,0.5237,0.2645,10252
1,DELIBERATELY LEAKY — discard,1.00,1.00,1.0000,1.0000,0.2645,10252


The ratio directly constructs the proxy: a high control score is leakage, not useful performance.


,Experiment,Precision@20,Precision@50,Average precision,ROC-AUC,Test base rate,Test pages
0,"Fixed LR, all four early features",0.50,0.48,0.2583,0.4799,0.2645,10252
1,"Fixed LR, early impressions removed",0.65,0.48,0.2609,0.4832,0.2645,10252


A change in this ablation measures feature dependence; it does not alone prove or disprove leakage.


,Fold,Feature,AP drop mean,Permutation SD,Test base rate
0,1,impressions_early,0.0406,0.0012,0.3546
1,1,avg_position_early,0.0486,0.0026,0.3546
2,1,position_volatility_early,0.0140,0.0026,0.3546
3,1,active_days_early,0.0217,0.0011,0.3546
4,2,impressions_early,0.0376,0.0025,0.4407
5,2,avg_position_early,0.0512,0.0039,0.4407
6,2,position_volatility_early,0.0278,0.0019,0.4407
7,2,active_days_early,0.0274,0.0004,0.4407
8,3,impressions_early,0.0563,0.0044,0.4446
9,3,avg_position_early,0.0754,0.0031,0.4446


,mean_ap_drop,min_fold_drop,max_fold_drop,mean_test_base_rate
Feature,,,,
avg_position_early,0.0499,0.0263,0.0754,0.3372
impressions_early,0.0366,0.0079,0.0563,0.3372
active_days_early,0.0187,0.0098,0.0274,0.3372
position_volatility_early,0.0167,0.0065,0.0283,0.3372


Negative AP drops mean shuffling helped in that fold; importances are diagnostic, not causal.
No features are changed in response to these outer-test diagnostics in this audit.


### Real failure examples

These **historical** examples are from the saved Week-5 output (cell headed “Three concrete
wrong picks”), not a rerun. All three were proxy negatives in the top-50 Logistic Regression queue.

| W5 queue rank | Pseudonymous content ID | Model score | Early impressions | Early average position | Early active days |
|---|---|---|---|---|---|
| 1 | `content_1c47c13983830602` | 0.712185 | 9,678 | 59.674797 | 14 |
| 4 | `content_bfb22aff4e9340f6` | 0.690285 | 791 | 59.766097 | 14 |
| 6 | `content_76d99ffa3d1b90d1` | 0.681601 | 483 | 56.269803 | 14 |

**What the evidence supports:** deep average position did not guarantee the defined later decline.
These are proxy-discordant picks, not proof that editorial review would have been useless.
The model used balanced class weights; its scores are ranking signals, not demonstrated calibrated
probabilities. Seasonality, query mix, and measurement noise are hypotheses for investigation,
not established explanations for these specific errors.

The next cell inspects new out-of-fold failures after the audit runs. It shows only a few
pseudonymous examples and aggregate queue errors. The label outcome is displayed for diagnosis
and is never fed into the honest model.

In [16]:
if checks["nested_cv"]:
    selected_queue = oof[oof.queue_rank <= 50].copy()
    queue_summary = selected_queue.groupby("fold").agg(
        reviewed=(TARGET, "size"), proxy_positives=(TARGET, "sum"), precision=(TARGET, "mean"))
    queue_summary["proxy_negative_picks"] = queue_summary.reviewed - queue_summary.proxy_positives
    queue_summary["test_base_rate"] = oof.groupby("fold")[TARGET].mean()
    display(queue_summary.round(4))
    false_picks = selected_queue[selected_queue[TARGET] == 0].sort_values(["queue_rank", "fold"])
    missed = oof[(oof.queue_rank > 50) & (oof[TARGET] == 1)].sort_values(["queue_rank", "fold"], ascending=[False, True])
    cols = ["fold", "queue_rank", "content_hash_id", "selected_model", "model_score", TARGET,
            "impressions_early", "impressions_late", "avg_position_early", "active_days_early"]
    print("NEW OOF EXAMPLES — up to three highest-priority proxy-negative picks:")
    display(false_picks[cols].head(3).round(4))
    print("NEW OOF EXAMPLES — up to three low-ranked proxy positives missed by the queue:")
    display(missed[cols].head(3).round(4))
    for label, examples in [("Proxy-negative pick", false_picks.head(3)), ("Missed proxy positive", missed.head(3))]:
        if examples.empty:
            print(f"No {label.lower()} observed; none invented.")
        for _, row in examples.iterrows():
            observed_change = (row.impressions_late / row.impressions_early - 1) * 100
            print(f"{label}: outer fold {int(row['fold'])}, rank {int(row.queue_rank)}, "
                  f"observed impression change {observed_change:+.1f}%. "
                  "This is an outcome-proxy mismatch; the cause is unverified.")
    # Fixed bands avoid learning slice cutoffs from test outcomes.
    oof["early_volume_band"] = pd.cut(oof.impressions_early,
        bins=[99, 500, 2000, np.inf], labels=["100–500", "501–2,000", ">2,000"])
    slices = []
    for band, frame in oof.groupby("early_volume_band", observed=True):
        queued = frame[frame.queue_rank <= 50]
        slices.append({"Early volume": str(band), "OOF pages": len(frame),
            "Slice base rate": frame[TARGET].mean(), "Queued pages": len(queued),
            "Queued proxy positives": int(queued[TARGET].sum()),
            "Queue precision": queued[TARGET].mean() if len(queued) else np.nan})
    display(pd.DataFrame(slices).round(4))
    print("Small slices may be unstable. Queue misses are capacity-based, not threshold-based false negatives.")
    checks["errors"] = True
else:
    print("NOT RUN — new OOF failures unavailable. The three markdown examples above are saved Week-5 observations only.")

,reviewed,proxy_positives,precision,proxy_negative_picks,test_base_rate
fold,,,,,
1,50,26,0.52,24,0.3546
2,50,37,0.74,13,0.4407
3,50,23,0.46,27,0.4446
4,50,30,0.60,20,0.2202
5,50,20,0.40,30,0.2259


NEW OOF EXAMPLES — up to three highest-priority proxy-negative picks:


,fold,queue_rank,content_hash_id,selected_model,model_score,is_declining_proxy,impressions_early,impressions_late,avg_position_early,active_days_early
41778,1,1,content_3e56218fa52d24b8,Random Forest,0.8381,0,4445.0,4619.0,47.5331,14
8950,3,1,content_57a7e6a1b4c862e2,Random Forest,0.8618,0,138.0,407.0,5.0524,13
51429,1,2,content_ceb1376cb6ade0db,Random Forest,0.8238,0,4928.0,4700.0,42.9683,14


NEW OOF EXAMPLES — up to three low-ranked proxy positives missed by the queue:


,fold,queue_rank,content_hash_id,selected_model,model_score,is_declining_proxy,impressions_early,impressions_late,avg_position_early,active_days_early
48588,1,17270,content_a56f819500637209,Random Forest,0.1321,1,477.0,9.0,3.0000,3
38329,1,17243,content_0d0036424bb6a712,Random Forest,0.1445,1,702.0,556.0,2.7620,9
51142,1,17225,content_ca8d327885b09a97,Random Forest,0.1509,1,705.0,383.0,2.6382,9


Proxy-negative pick: outer fold 1, rank 1, observed impression change +3.9%. This is an outcome-proxy mismatch; the cause is unverified.
Proxy-negative pick: outer fold 3, rank 1, observed impression change +194.9%. This is an outcome-proxy mismatch; the cause is unverified.
Proxy-negative pick: outer fold 1, rank 2, observed impression change -4.6%. This is an outcome-proxy mismatch; the cause is unverified.
Missed proxy positive: outer fold 1, rank 17270, observed impression change -98.1%. This is an outcome-proxy mismatch; the cause is unverified.
Missed proxy positive: outer fold 1, rank 17243, observed impression change -20.8%. This is an outcome-proxy mismatch; the cause is unverified.
Missed proxy positive: outer fold 1, rank 17225, observed impression change -45.7%. This is an outcome-proxy mismatch; the cause is unverified.


,Early volume,OOF pages,Slice base rate,Queued pages,Queued proxy positives,Queue precision
0,100–500,35334,0.3280,124,80,0.6452
1,"501–2,000",25699,0.3296,29,11,0.3793
2,">2,000",14493,0.3767,97,45,0.4639


Small slices may be unstable. Queue misses are capacity-based, not threshold-based false negatives.


## 4. Claim rewrite

| Earlier wording in my Week-5 notebook | Why I narrow it | Public-safe rewrite |
|---|---|---|
| “twice as many useful picks within the review limit” | The measured positives are declines, not verified editorial benefit, and the winner was selected on the test queue. | In the saved March client-held-out development run, Logistic Regression ranked 24 proxy-positive pages in its top 50, compared with 12 for the rule. This observed difference supports further investigation as decision support; it does not establish twice the editorial value. |
| “The model made 26 wrong picks” | A proxy-negative page might still deserve review for another reason. | Of the saved top-50 LR picks, 26 did not meet the defined later-impression-decline proxy. No editorial usefulness label was collected. |
| “Random Forest … additional complexity was not justified” | A single selected holdout does not establish a general model ranking. | Random Forest had lower measured Precision@50 on that development holdout (0.28 versus 0.48 for LR); nested grouped validation tests whether this ordering is stable within March. |
| “rank pages by their probability” | Balanced weights and no calibration test do not justify treating scores as event probabilities. | Use model scores to order a review queue; calibrated probabilities and real editorial outcomes remain unverified. |

The new claim below is generated only from successfully computed audit results. If the model
does not beat the rule, that outcome is retained. No refresh effect, production readiness,
Google ranking mechanism, or generalization to future months is claimed.

In [17]:
if checks["nested_cv"] and checks["leakage"] and checks["errors"]:
    selected = cv_summary.set_index("Method").loc["Inner-selected model"]
    baseline = cv_summary.set_index("Method").loc["Week-4 baseline rule"]
    delta = selected.mean_p50 - baseline.mean_p50
    verdict = "exceeded" if delta > 0 else "fell below" if delta < 0 else "matched"
    claim = (
        f"On {len(model_data):,} eligible pages from {model_data[GROUP].nunique()} pseudonymized clients "
        "in March 2026, we measured the two fixed Week-5 model families using five outer client-disjoint "
        "folds, selecting the model within three inner grouped folds. "
        f"The inner-selected model had mean Precision@50 {selected.mean_p50:.3f} "
        f"(fold range {selected.min_p50:.3f}–{selected.max_p50:.3f}), versus "
        f"{baseline.mean_p50:.3f} for the training-fitted rule and "
        f"{selected.mean_test_base_rate:.3f} mean held-out prevalence. "
        f"Its mean P@50 {verdict} the rule by {abs(delta):.3f}. "
        f"Mean average precision was {selected.mean_ap:.3f}; mean ROC-AUC was {selected.mean_auc:.3f}. "
        "These are directional development observations for editorial decision support under a "
        "future-impression-decline proxy. Each outer fold supplies its own 50-page queue. "
        "The audit does not measure editorial benefit, a causal refresh effect, probability calibration, "
        "or performance in a new month."
    )
    display(Markdown(claim))
    checks["claim"] = True
else:
    display(Markdown("**New empirical claim: pending.** The new grouped audit has not completed. "
                     "Only the explicitly labeled saved Week-5 observations above can currently be reported."))

On 75,526 eligible pages from 37 pseudonymized clients in March 2026, we measured the two fixed Week-5 model families using five outer client-disjoint folds, selecting the model within three inner grouped folds. The inner-selected model had mean Precision@50 0.544 (fold range 0.400–0.740), versus 0.504 for the training-fitted rule and 0.337 mean held-out prevalence. Its mean P@50 exceeded the rule by 0.040. Mean average precision was 0.406; mean ROC-AUC was 0.581. These are directional development observations for editorial decision support under a future-impression-decline proxy. Each outer fold supplies its own 50-page queue. The audit does not measure editorial benefit, a causal refresh effect, probability calibration, or performance in a new month.

## 5. Self-check

The final table distinguishes completed writing from completed experiments. An error-free run
that prints `NOT RUN` is **not** an executed validation audit. New metrics and failure examples
must be present before submission. No checkbox claims that a model rerun occurred merely because
the code was prepared.

After all computed checks pass, review the paper questions, outputs, and claim, save the executed
notebook to GitHub at `work/notebooks/w06_validation_audit.ipynb`, and verify its rendered outputs.
Submit the repository URL on the assignment card:
[github.com/Davidsoooooon/ML-Intern---Briones](https://github.com/Davidsoooooon/ML-Intern---Briones).

No datasets, credentials, raw queries, client names, or client URLs belong in the commit.
Only the notebook with aggregate metrics and the limited pseudonymous diagnostic examples is needed.

In [18]:
check_rows = [
    ("Two paper findings, page references, and constructive questions", True),
    ("Correct Week-5 method and historical evidence identified", True),
    ("Approved March data loaded and matching cohort verified", checks["data"]),
    ("Raw grain and model-frame integrity verified", checks["grain"]),
    ("New original-design and random/grouped comparisons computed", checks["comparison"]),
    ("Nested client-disjoint selection/evaluation completed", checks["nested_cv"]),
    ("Leakage control, ablation, and held-out importance completed", checks["leakage"]),
    ("New out-of-fold failures inspected", checks["errors"]),
    ("Public-safe claim generated from new results", checks["claim"]),
]
display(pd.DataFrame([{"Check": name, "Status": "PASS" if ok else "PENDING"}
                      for name, ok in check_rows]))
if all(checks.values()):
    print("COMPUTATION COMPLETE — review the outputs, save this executed notebook to GitHub, then submit the repo URL.")
else:
    print("NOT SUBMISSION-READY — warehouse-dependent experiments remain incomplete. Enable the existing Colab HF_TOKEN secret and Run all.")
print("Human review, saving the executed version, and portal submission are not asserted by this cell.")

,Check,Status
0,"Two paper findings, page references, and const...",PASS
1,Correct Week-5 method and historical evidence ...,PASS
2,Approved March data loaded and matching cohort...,PASS
3,Raw grain and model-frame integrity verified,PASS
4,New original-design and random/grouped compari...,PASS
5,Nested client-disjoint selection/evaluation co...,PASS
6,"Leakage control, ablation, and held-out import...",PASS
7,New out-of-fold failures inspected,PASS
8,Public-safe claim generated from new results,PASS


COMPUTATION COMPLETE — review the outputs, save this executed notebook to GitHub, then submit the repo URL.
Human review, saving the executed version, and portal submission are not asserted by this cell.
